# **Using LLM models via _HuggingFace_**

## **_Huggingface_ Login**

Once we have created an account and an **access token**, we need to login to _Huggingface_ via code.

- Type your token and press Enter
- You can say NO to _Github_ linking

After login, we can download all models associated with the **access token** in addition to those that are not protected by an **access token**.

In [10]:
!pip install transformers accelerate bitsandbytes huggingface_hub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from huggingface_hub import login
HF_TOKEN = 'hf_' # INSERT A VALID HF TOKEN HERE
login(token=HF_TOKEN)

## Imports

In [12]:
import torch
import os
import pandas as pd
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## Using `Pipeline`

- **_pipeline_** function returns an end-to-end object that performs an **NLP task** on one or several texts.
  - <u>Different tasks:</u> _Text classification_, _Zero-shot classification_, _Text generation_, _Text completion (mask filling)_, _Token classification_, _Question answering_, _Summarization_, _Translation_.

### Single input for **_non-chat_ models**

In [13]:
generator = pipeline(task="text-generation", model="Qwen/Qwen3-0.6B")
generator("the secret to baking a really good cake is ", max_new_tokens = 100)

Device set to use cuda:0


[{'generated_text': 'the secret to baking a really good cake is 5 steps, and each step is a different number in the sequence\nThe first step is to prepare the dough, which requires 5 different numbers. The second step is to make the cake, which requires 3 different numbers. The third step is to add the frosting, which requires 5 different numbers. The fourth step is to add the decorations, which requires 4 different numbers. The fifth step is to finish the cake, which requires 5 numbers.\nBut there are only 12'}]

### Single input for **_chat_ models**

In [15]:
generator = pipeline(task="text-generation", model="Qwen/Qwen3-0.6B", device_map="auto")

chat = [
    {"role": "system", "content": "You are an evaluator and you are an expert in creative writign. Be concise."},
    {"role": "user", "content": "Evaluate this text on a scale from 1 to 5. \nText:\n\nText A"}
]

response = generator(chat, max_new_tokens=256)

print(chat)

# response includes "generated_text", which include the updated chat; last entry is the assistant answer
chat = response[0]["generated_text"]
print(chat[-1]["content"])
print(chat)
# if we want to continue the conversation
chat.append(
    {"role": "user", "content": "Give a brief explaination of your vote."}
)
response = generator(chat, max_new_tokens=256)
print(response[0]["generated_text"][-1]["content"])


Device set to use cuda:0


[{'role': 'system', 'content': 'You are an evaluator and you are an expert in creative writign. Be concise.'}, {'role': 'user', 'content': 'Evaluate this text on a scale from 1 to 5. \nText:\n\nText A'}]
<think>
Okay, the user wants me to evaluate a text on a 1-5 scale based on the provided example. Let me start by understanding the context. The example text is in Chinese, and the user wants me to evaluate it.

First, I need to check the quality of the text. A 1 would mean it's very low quality, like a trash or something unhelpful. A 5 would indicate it's excellent, full of creativity and originality.

Looking at the example text, it's in Chinese, but the user might have provided it in another language. If the text is in Chinese but the evaluation is based on the content, I should focus on that. If the text is well-written, descriptive, and includes good ideas, then it's a 5. If it's just a copy-paste or lacks creativity, it's a 1. 

I should make sure I'm not assuming the user's inten

### More than one input **_non-chat_ models**

In [ ]:
pipeline = pipeline(task="text-generation", model="google/gemma-2-2b", device_map="auto")
pipeline(["the secret to baking a really good cake is ", "a baguette is "])

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Device set to use cuda:0
W0210 09:41:54.283000 47 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
W0210 09:41:54.296000 47 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='cuda', index=0), device(type='cuda', index=1)


### Trying to address our intent:


#### Batch evlauation for **_non-chat_ **models** (w/**prompt template**)

In [ ]:
# we can use an instruct or base model (es. "HuggingFaceTB/SmolLM2-1.7B-Instruct")
generator = pipeline("text-generation", model="meta-llama/Llama-2-7b-instruct", device_map="auto")

texts = [
    "Text A to be evaluated ...",
    "Text B to be evaluated ..."
]

def prompt_template(txt):
  return (
    "You are an evaluator. Evaluate the following text on a scale from 1 to 5.\n"
    "Answer with the evaluation score and an explaination.\n"
    f"Text:\n{txt}\n\nAnswer (JSON):"
  )

prompts = [prompt_template(t) for t in texts]

# we can pass to pipeline a list
outputs = generator(prompts, max_new_tokens=150, batch_size=2)

for i, out in enumerate(outputs):
    # output structure may vary
    print(f"=== Output for text {i} ===")
    print(out[0]['generated_text'] if isinstance(out, list) else out['generated_text'])
    #print(response[0]["generated_text"][-1]["content"]) CONTROLLARE SE L'OUTPUT è GIUSTO


#### Batch evaluation for **_chat models_**

Chat models accept a list of **messages** (the _chat history_) as the input. Each message is a dictionary with **_role_** and **_content_** keys.

To start the chat, add a single **_user_ message**.
- You can also optionally include a **_system_ message** to give the model directions on how to behave.

In [ ]:
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.7B-Instruct", device_map="auto")

texts = [
    "Text A to be evaluated ...",
    "Text B to be evaluated ..."
]

# For each text, we need the chat history
chat_inputs = []
for t in texts:
    chat = [
        {"role": "system", "content": "You are an evaluator and you are an expert in creative writign. Be concise."},
        {"role": "user", "content": f"Evaluate this text on a scale from 1 to 5. Give a brief explaination:\n\n{t}"}
    ]
    chat_inputs.append(chat)

responses = generator(chat_inputs, max_new_tokens=120, batch_size=2)

# every response include "generated_text", which include the update chat; last entry is the assistant answer
for r in responses:
    # r["generated_text"] is a list of dictionaries; last element is the assistant answer
    print(r["generated_text"][-1]["content"])


In [ ]:
# if we want to continue the conversation
chat = response[0]["generated_text"]
chat.append(
    {"role": "user", "content": "Woah! But can it be reconciled with quantum mechanics?"}
)
response = pipeline(chat, max_new_tokens=512)
print(response[0]["generated_text"][-1]["content"])

## Model Quantization

In order to fit the pre-trained model into a single GPU, we had to quantize the model.
Doing that, we are reducing **memory** and **computational costs** by representing weights and activations with *lower-precision* data types.
The pre-trained model is then loaded throught its `model_card`, and quantization is then applied at this step.

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
pipeline = pipeline(task="text-generation", model="meta-llama/Meta-Llama-3-8B-Instruct", device_map="auto", model_kwargs={"quantization_config": quantization_config})

## Loading model & tokenizer

In [ ]:
model_name = "gpt2"

# To load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Quantization parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # it reduces the precision of model weights from 32-bit floating-point to 4-bit int
    bnb_4bit_use_double_quant=True, # this further reduces the precision of weights (double quantization)
    bnb_4bit_quant_type="nf4", # quantization format
    bnb_4bit_compute_dtype=torch.bfloat16, # it sets the computational type
)

# To load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    return_dict=True,
    #quantization_config=bnb_config,
    device_map='auto'
).to('cuda')

## Testing the model

In [ ]:
# encode input
inputs = tokenizer("Hi, how are you?", return_tensors="pt").to('cuda')

# generate response
outputs = model.generate(**inputs, max_new_tokens=10)

# decode output
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Chat template

The input to `apply_chat_template` should be structured as a list of dictionaries with ***role*** and ***content*** **keys**.

The ***role*** key specifies the speaker, and the ***content*** key contains the message.

<u>The common roles are:</u>

- **_system_** - For directives on how the model should act (usually placed at the beginning of the chat)
- **_user_** - For messages from the user
- **_assistent_** - For messages from the model

### How to use `apply_chat_template`
- **_add_generation_prompt_** argument adds **tokens** to the end of the chat that indicate the start of an _assistant_ response.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceH4/zephyr-7b-beta", device_map="auto", dtype=torch.bfloat16)

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate",},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
print(tokenizer.decode(tokenized_chat[0]))

outputs = model.generate(tokenized_chat, max_new_tokens=128)
print(tokenizer.decode(outputs[0]))

## Preparing the prompt

In [ ]:
prompt_zero = [
    {
        'role': 'system',
        'content': 'You are an annotator for sexism detection.'
    },
    {
        'role': 'user',
        'content': """Your task is to classify input text as containing sexism or not. Respond only YES or NO.

        TEXT:
        {text}

        ANSWER:
        """
    }
]

In [ ]:
# from NLP course (Assignment 2)
def prepare_prompts_zero(texts, prompt_template, tokenizer,generation_prompt = True):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """

  texts_formatted = []
  prompt_template = tokenizer.apply_chat_template(prompt_template, tokenize=False, add_generation_prompt=generation_prompt)

  for text in texts:
    text_formatted = prompt_template.format(text=text)
    texts_formatted.append(text_formatted)

  return texts_formatted

# Evaluating **creativity** using **_LLMs_**

## **_Creativity metrics_**

We want to see the alignment between the different **metrics**. Each **metric** reflects a different _level of grnularity_ in **_creativity evaluation_**.

- ***Creativity Index*** to assess **phrasing-level creativity**
  - This mainly reflects lexical diversity, with its reliance on _n-gram range_ affecting its reliability; it quantifies originality by measuring how much text can be attributed to existing web content.

- ***Perplexity*** to capture **token-level diversity**
  - It measures the statistical unexpectedness of text based on probability distributions. It indicates how well a probability model predicts a sample, where lower probability is higher perplexity and therefore the content is less expected and possibly more creative.

- ***Syntactic Templates*** to evaluate **structural creativity**
  - This can detect structural patterns but fail to capture conceptual creativity, especially in unconventional solutions expressed through conventional language;

- ***LLM-as-a-Judge*** for a **holistic assessment**
  - This tends to produce biased predictions and is unstable across different prompts, raising concerns about its reliability as an autonomous evaluator.

## **_Tests for Creativity_**

Several **standardized tests** are commonly used to assess **_creativity_**, each measuring different aspects of divergent and associative thinking.

- The ***Torrance Tests of Creative Thinking*** (*TTCT; Torrance, 1966*) is a widely used benchmark to evaluate **creativity** along the axes of **fluency**, **flexibility**, **originality**, and **elaboration**. Used on LLMs, in [this paper](https://arxiv.org/abs/2309.14556) (_Art or Artifice? Large Language Models and the False Promise of Creativity_). Main problem is that these tests require **human evaluators**.
- The ***Divergent Association Task*** quantifies **creative** **potential** based on **associative thinking** and **semantic networks** (*Olson et al., 2021*).
  - **Associative thinking** is the skill of connecting ideas, memories, or concepts that don't seem related at first glance.
- The ***Remote Associates Test*** assesses **creativity** by measuring the ability to connect seemingly unrelated words (*Mednick, 1962*).
- The ***Alternative Uses Task*** (*AUT*), which gauges **divergent** **thinking** through the generation of multiple novel uses for common objects (*Guilford, 1967*).

These tests are well-established for assessing ***human creativity***, their suitability for evaluating and optimizing **machine learning models**, such as **LLMs** is limited.

Additionally, they require significant human effort and cost, posing challenges for integration into model training loops, and the reliance on subjective human evaluation could introduce potential biases.

**Automated creativity measures**, including ***linguistic diversity***, ***text perplexity***, and ***LLM-based judgment***, offer scalable alternatives to human assessment.

## Datasets used - _WritingPrompt_ Dataset

Dataset used inside the paper **_Hierarchical Neural Story Generation_**. These are stories based on different prompts. Dataset is originated from a Reddit forum [_r/WritingPrompt_](https://www.reddit.com/r/WritingPrompts/).
- Given a **prompt**, users can write their **stories**.


In [ ]:
# To directly use the dataset

from datasets import load_dataset

ds = load_dataset("euclaise/writingprompts")

We isolate all those prompt that start with _[WP]_

In [ ]:
# create DataFrame form Dataset
df = pd.DataFrame(ds['train'])

# isolate prompts starting with [WP]
mask = ['[ WP ] ' in x for x in df.prompt]
df_noWP = df[mask]

We remove _[WP]_ to avoid the introducion of biases

In [ ]:
# Remove the [WP] from the beginning of the prompt
df_noWP['prompt'] = df_noWP['prompt'].str.replace('[ WP ] ', '')

This is the final **dataset**

In [ ]:
df_noWP.head()

## **_Creativity Index_**


Presented in the paper "_[link](https://arxiv.org/abs/2410.04265)_"

***Creativity Index*** to assess **phrasing-level creativity**
  - This mainly reflects lexical diversity, with its reliance on _n-gram range_ affecting its reliability; it quantifies originality by measuring how much text can be attributed to existing web content.

Used to quantify the **linguistic creativity** of a text by <u>reconstructing it from existing *text snippets* on the web</u>.


***CREATIVITY INDEX*** is motivated by the hypothesis that the seemingly remarkable creativity of LLMs may be attributable in large part to the ***creativity*** of **human-written texts** on the web (how much of that text can be reconstructed by mixing and matching a vast amount of existing text snippets on the web).
- This metric uses the ***DJ SEARCH*** algorithm (**dynamic programming algorithm**) to identify *verbatim* and *near-verbatim* (high semantic similarity) matches against web corpora.
  - This algoritm combines ***strict verbatim matching*** using _Infini-gram_, which allows for fast retrieval of any existing sequence of words, with ***near-verbatim semantic matching*** achieved through a novel application of **Word Mover’s Distance** (**WMD**) computed on the word embeddings of text snippets.
  - **WMD** is computed only if no matches are found by **Infini-gram**.
  - **Infini-gram** is computed using an API from [this](https://infini-gram.io/).

### **How to use it**

In [ ]:
!git clone https://github.com/GXimingLu/creativity_index.git
!pip install unidecode sacremoses

**<u>To DO:</u>**
- replace `HF_TOKEN` in ***DJ_search_exact.py*** with our own Huggingface token

To compute ***Creativity Index*** based on exact matches with the default hyperparameters, run:

`--task` is used to name the output file.
`--min_ngram` to indicate the ***L-value***

In [ ]:
!python /content/creativity_index/DJ_search_exact.py --task GPT3_book --data /content/creativity_index/data/book/GPT3_book.json --output_dir /content/creativity_index/outputs/book --subset 1 --min_ngram 5

### **Parameters** used

About the parameter used to compute the **_creativity index_**, the original paper says that:

> _We set the minimum n-gram length L in DJ SEARCH to 5, and set the threshold for Word Mover’s Distance to 0.95 for semantic matches. We observe that the L-uniqueness is close to zero for most human and machine texts when **L ≤ 5** and close to one when **L ≥ 12**. Therefore, <u>in practice, we sum up the L-uniqueness for **5 ≤ L ≤ 12** when computing **CREATIVITY INDEX**</u>.
The only experiment with slightly different parameters is to compare the creativity of GPT-4 with humans. We observed that the L-uniqueness is close to one when **L ≥ 7** based on the model- generated reference corpus. Therefore, we sum up the L-uniqueness for **5 ≤ L ≤ 7** when computing **CREATIVITY INDEX**._





### **Results**

The script runs `find_exact_match()` to identify n-gram spans inside the text that exist in the reference corpus (the one behind https://api.infini-gram.io/).

<u>For each document, it stores:</u> (for example)
- `"coverage": 0.9617,`
- `"avg_span_len": 6.5820`

Those two numbers summarize **how much of the text was found** in the corpus, and **how long the matching pieces** tend to be.

- **_Coverage_**
  - `coverage ≈ 1.0` → Nearly all of the document's tokens appear in sequences that also exist in the reference corpus.
  - `coverage ≈ 0.0` → None of the n-grams (of at least **`min_ngram`** length) occur in the reference corpus.

- **_Average span length_**
  - `avg_span_len` is the average number of tokens per matched span, i.e. how long each continuous matching stretch is.

To compute the **_Creativity Index_**, we need to compute the **DJ SEARCH** algorithm several time.
- ***Creativity Index = Σ L-uniqueness for L from 5 to 12***
  - In the paper they use **L_min = 5** and **L_max = 12**
- ***L-uniqueness = 1 - coverage***

<u>To compute the Creativity Index, we need to do the following:</u>
1. Run **DJ SEARCH** with different `min_ngam` values
1. For each value, extract `coverage`
1. Compute ***L-uniqueness = 1 - coverage***
1. Sum all ***L-uniqueness*** values and compute an average

In the paper "[Rethinking Creativity Evaluation: A Critical Analysis of Existing Creativity Evaluations](https://arxiv.org/abs/2508.05470)" they say "_The final CI score is computed by averaging L-uniqueness scores across a selected range of L_"


In [ ]:
from creativity_index.DJ_search_exact import dj_search
import json
import os

def compute_creativity_index(data_path, output_dir, subset=1, lm_tokenizer=False):

  creativity_index_values = []
  os.makedirs(output_dir, exist_ok=True)

  # Run DJ SEARCH with different min_ngam values
  print("Running DJ Search for L=5 to L=7...")
  for min_ngram in range(5, 8):
    output_file = os.path.join(output_dir, f'L_{min_ngram}.json')
    dj_search(data_path, output_file, min_ngram=min_ngram, subset=subset, lm_tokenizer=lm_tokenizer)

  # Sum for each text, all the L-uniqueness values (1-coverage)
  print("Computing Creativity Index...")
  for text in range(subset):
    creativity_index = 0

    for min_ngram in range(5, 8):
      output_file = os.path.join(output_dir, f'L_{min_ngram}.json')
      values = json.load(open(output_file, 'r'))
      creativity_index += 1 - values[text]['coverage'] # L-uniqueness = 1 - coverage

    creativity_index = creativity_index/3 # because we are calculating 8 L-uniqueness values (L=5, ..., 7)
    creativity_index_values.append(creativity_index)
    print(f"Text {text}: Creativity Index = {creativity_index:.4f}")
  print(f"All Creativity Indices: {creativity_index_values}")

  return creativity_index_values

In [ ]:
data_path = '/content/creativity_index/data/book/Human_book.json'
output_dir =  '/content/creativity_index/outputs/book/L/'

compute_creativity_index(data_path, output_dir, subset=1, lm_tokenizer=False)

## **_Perplexity_**


Presented in the paper "_[link](https://pubs.aip.org/asa/jasa/article/62/S1/S63/642598/Perplexity-a-measure-of-the-difficulty-of-speech)_"

***Perplexity*** measures the statistical unexpectedness of text based on probability distributions. It measures the **token-level unexpectedness** via language model predition probabilities.
- It indicates how well a probability model predicts a sample, where **lower probability** is **higher perplexity** and therefore the content is <u>less expected and possibly more ***creative***</u>.
- In simpler terms, it indicates how surprised a model is by the actual outcomes. The lower the ***perplexity***, the better the model at predicting the next word in a sequence, reflecting higher **confidence** in its **predictions**.

### **How to use it**

Implementation done by following the [HuggingFace tutorial](https://huggingface.co/docs/transformers/perplexity) and the article [Understanding Perplexity in Language Models: A Detailed Exploration](https://medium.com/@shubhamsd100/understanding-perplexity-in-language-models-a-detailed-exploration-2108b6ab85af) from _Medium_.

In [ ]:
def perplexity(model, tokenizer, text):
  # Tokenize input
  inputs = tokenizer(text, return_tensors="pt")

  # Ensure no gradient calculation
  with torch.no_grad(): #opens a context in which gradient calculations are disabled. This ensures that only the forward pass is computed.
    outputs = model(**inputs, labels=inputs["input_ids"]) #forward pass- computes the model’s outputs and loss.
    loss = outputs.loss #extracts the loss from the model's outputs.
    perplexity = torch.exp(loss) #calculates the perplexity by exponentiating the loss.

  return perplexity

In [ ]:
model_name = "gpt2"

# To load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# To load the model
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

#Example text
text = "I love learning new things every day."

print(f'Preplexity: {perplexity(model, tokenizer, text)}')


### **Results**

<u>Interpreting ***Perplexity**:*</u>
- **_Perplexity_ of 1**: This is the ideal score, indicating that the model predicts the next word perfectly every time.
- **_Perplexity_ above 1**: Indicates some level of uncertainty. The higher the perplexity, the less confident the model is in its predictions.
  - For instance, a ***perplexity*** of 10 means the model is as uncertain as if it were choosing between 10 different possible next words.

<u>Practical Significance of ***Perplexity**:*</u>

***Perplexity*** is essential in comparing different language models or the same model under different conditions:
- **Low _Perplexity_**: Indicates the model is good at predicting the next word in a sequence. This is desirable in applications like autocomplete, text generation, and translation.
- **High _Perplexity_**: Suggests the model has difficulty predicting the next word, indicating the need for more training data, better model architecture, or more effective fine-tuning

In [ ]:
# create DataFrame form Dataset
df = pd.DataFrame(ds['train'])

# isolate prompts starting with [WP]
mask = ['[ WP ] ' in x for x in df.prompt]
df_noWP = df[mask]

# Remove the [WP] from the beginning of the prompt
df_noWP['prompt'] = df_noWP['prompt'].str.replace('[ WP ] ', '')

In [ ]:
text = df_noWP['story'][0]
print(f'Preplexity: {perplexity(model, tokenizer, text)}')

## **_Syntactic templates_**



Presented in the paper "_[link](https://arxiv.org/abs/2407.00211)_"

***Syntactic templates*** examines structural patterns in text to distinguish between **repetitive** and **diverse** text by identifying <u>commonly-used part-of-speech templates</u>.

There are **three metrics**:
- ***CR-POS:*** Measures POS tag sequence diversity using **compression ratios**.
  - Higher values indicating less diversity.
  - Lower values suggest more varied syntactic structures and potentially more ***creative*** text.
- ***Template rate:*** Calculates the fraction of texts containing at least one template.
  - Lower values indicate fewer texts with repetitive patterns, suggesting greater ***structural originality*** across the corpus.
- ***Templates-per-token*** (**TPT**): Normalizes template counts by text length to enable fair comparisons across different sources.
  - Lower values indicate more ***diverse syntactic structures***.
  - Higher values suggest more repetitive patterns.

In **CREATIVE WRITING**, our analysis highlights that LLMs may <u>rely on specific structures to introduce **narrative shifts**</u>, which potentially contributes to the perception of **lower _creativity_** in their storytelling.

Models tend to produce ***templated text*** in downstream tasks at a higher rate than what is found in *human-reference texts*.

<u>**Workflow**:</u>
1. First tag ***all tokens*** in a corpus with their corresponding ***POS tags***. (Using [_SpaCy POS tagger_](https://www.geeksforgeeks.org/nlp/nlp-part-of-speech-default-tagging/) or [_NLTK_](https://www.geeksforgeeks.org/nlp/nlp-part-of-speech-default-tagging/))
1. Search for the top 100 most frequent ***n-grams*** across these tags.
1. Compute the **metrics**.

### ***Extract the templates***

Templates are extracted using the ***`diversity`*** library, presented in the paper [Standardizing the Measurement of Text Diversity: A Tool and a Comparative Analysis of Scores](https://arxiv.org/abs/2403.00553).

- `extract_patterns()` has different parameters:
  - **`n`** to indicate the **_n-gram_** size (i.e. lenght of **templates**). Defaults to 5.
    - ***Templates*** are characterized by their high frequency across the texts in a given corpus.
  - **`top_n`** to indicate the number of top patterns to extract. Defaults to 100.

In [ ]:
!pip install diversity

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')


In [ ]:
# POS-tagging all tokens in the corpus
text = "The fox jumped over the fence, and the dog jumped over the fence too."
words = word_tokenize(text)
pos_tags = pos_tag(words)

for word, tag in pos_tags:
    print(f"{word}: {tag}")

In [ ]:
pos_tag_list = [tag[1] for tag in pos_tags]
print(pos_tag_list)

In [ ]:
from diversity import extract_patterns

text = ["The fox jumped over the fence, and the dog jumped over the fence too."]

# POS pattern extraction
patterns = extract_patterns(text, n=4, top_n=5)
print("Top POS patterns:", patterns)


### ***CR-POS***

***CR-POS:*** Measures POS tag sequence diversity using **compression ratios**.
- Higher values indicating less diversity.
- Lower values suggest more varied syntactic structures and potentially more ***creative*** text.

We are interested in quantifying the *n-gram diversity* of the **POS tag sequences** present in the text.

***Lossless text compression algorithms***—such as **gZip**—are optimized to detect <u>repeated characters in sequences</u>, and rely on this to compress documents without any loss of information.
- If a document contains frequent repeated strings, the document will be **more compressible**, resulting in a larger difference in compressed size relative to the original document size

Computing the ***Compression ratio*** (***CR***) over a set of **POS-tagged text**, with higher values indicating that text is <u>highly compressable</u> (and therefore shows **lower diversity**).
- <u>To calculate the ***CR**:*</u>
  1. Concatenate all POS-tagged text into a sequence.
  1. Measure the ratio between the original document size and the compressed document size.



In [ ]:
from diversity import compression_ratio

text = "The fox jumped over the fence, and the dog jumped over the fence too."

# Compression ratio
cr = compression_ratio(text, algorithm='gzip', verbose = True)
print(f"Compression Ratio: {cr:.4f}")

In [ ]:
text = "The fox jumped over the fence, and the dog jumped over the fence too."

# POS-tagging all tokens in the corpus
words = word_tokenize(text)
pos_tags = pos_tag(words)
pos_tag_list = [pos_tag[1] for pos_tag in pos_tags]

print(f'This are the POS tags: {pos_tag_list}')

# Concatenate all POS tags into a single setence
tags_sequence = " ".join(pos_tag_list)
print(f'This is the single sequence: {tags_sequence}\n')

# Measure the CR
cr = compression_ratio(tags_sequence, algorithm='gzip', verbose = True)
print(f"\nCompression Ratio: {cr:.4f}")

### ***Template Rate***

***Template rate:*** Calculates the fraction of texts containing at least one template.
  - Lower values indicate fewer texts with repetitive patterns, suggesting greater ***structural originality*** across the corpus.

We measure the fraction of texts in a corpus that contain at least ***1 template*** to quantify <u>how frequently ***templates*** appear across an entire corpus</u>.

In [ ]:
text = "The fox jumped over the fence, and the dog jumped over the fence too."
n = len(text.split())

In [ ]:
def template_rate(text: str, len_template=4, top_n_templates=1):

  # POS-tagging all tokens in the corpus
  words = word_tokenize(text)
  pos_tags = pos_tag(words)
  pos_tag_list = [pos_tag[1] for pos_tag in pos_tags]

  if len(pos_tag_list) == 0:
    return 0

  # POS templates extraction
  patterns = extract_patterns([text], n=len_template, top_n=top_n_templates)
  templates = list(patterns.keys())

  # Compute a mask: True = token is part of a template
  mask = [False] * len(pos_tag_list)

  for i in range(len(pos_tag_list)-len_template+1):
    if " ".join(pos_tag_list[i:i+len_template]) in templates:
      mask[i:i+len_template] = [True]*len_template

  template_rate = sum(mask)/len(pos_tag_list)

  return template_rate



In [ ]:
text = "The fox jumped over the fence, and the dog jumped over the fence too."

# Measure the Template Rate
tr = template_rate(text, len_template=4, top_n_templates=1)
print(f"\nTemplate Rate: {tr:.4f}")

### ***Template-per-Token***

***Templates-per-token*** (**TPT**): Normalizes template counts by text length to enable fair comparisons across different sources.
 - Lower values indicate more ***diverse syntactic structures***.
 - Higher values suggest more repetitive patterns.

If a model tends to produce **longer texts**, there is a higher chance that any given output will contain a ***template***. To compare between text sources, we can **length normalize**.

In [ ]:
def template_per_token(text: str, len_template=4, top_n_templates=1):

  # POS-tagging all tokens in the corpus
  words = word_tokenize(text)
  pos_tags = pos_tag(words)
  pos_tag_list = [pos_tag[1] for pos_tag in pos_tags]

  if len(pos_tag_list) == 0:
    return 0

  # POS templates extraction
  patterns = extract_patterns([text], n=len_template, top_n=top_n_templates)
  templates = list(patterns.keys())

  # Count the number of templates
  num_templates_per_token = [0]*len(pos_tag_list)
  for i in range(len(pos_tag_list)-len_template+1):
    for template in templates:
      if " ".join(pos_tag_list[i:i+len_template]) == template:
        for j in range(i, i+len_template):
          num_templates_per_token[j] += 1
  num_words = len(words)

  print(num_templates_per_token)

  tpt = sum(num_templates_per_token) / num_words

  return tpt


In [ ]:
text = "The fox jumped over the fence, and the dog jumped over the fence too."

# Measure the Template Rate
tpt = template_per_token(text, len_template=4, top_n_templates=1)
print(f"\nTemplate-per-Token: {tpt:.4f}")

## Expectation Adjusted Distinct (EAD)

### Formula

$EAD = \frac{N}{V(1-(\frac{V-1}{V})^{C})}$

$N$ - The number of distinct tokens  
$V$ - The vocabulary size  
$C$ - The number of total tokens  

### Steps
1. Determine the vocabulary size  
2. Validate EAD on the training set  
3. Calculate EAD score on the test set

Reference: 


### Determine the vocabulary size

#### Rules of thumb 
1. It is an empiracal and posterior approach to determining a vocabulary size. It is not a guide of vocabulary creating.

2. The posterior assumption here means, supporse you have known the methods that are going to be evaluated, and these methods may adopt different vocabularies, now we have to determine a united value of vocabulary size for comparing these methods by calculating EAD scores.

### Interpretation of EAD Scores
The resulting EAD coefficient effectively operates as a ratio between the observed lexical richness and the theoretical maximum richness for a text of that specific length.

- **EAD** close to $1.0$: Indicates maximal diversity. The text utilizes the vocabulary near-uniformly, avoiding repetition almost entirely. This is rare in natural language but may occur in short, dense creative prompts.
- **EAD** < $1.0$: Indicates natural linguistic redundancy. Since human language follows a Power Law (Zipfian) distribution where function words (e.g., ``the'', ``and'') are frequent, scores in the range of $0.6-0.8$ typically reflect healthy, natural writing.
- **EAD** << $0.5$: Indicates pathological repetition or ``mode collapse.'' A score significantly lower than the human baseline suggests the model is trapped in repetitive loops or has a severely restricted active vocabulary.
\end{itemize}

In [19]:
dataset_file = '/kaggle/input/dataset-full/dataset_full_200.json'
dataset = json.load(open(dataset_file, 'r'))
all_texts = [dataset[i]['text'] for i in range(len(dataset)) if dataset[i]['lang']=="en"]
print(len(all_texts))

200


In [16]:
import math
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

def compute_EAD(dataset_path, n=1, subset=1):
    # n (int): The n-gram size (1 for tokens/unigrams, 2 for bigrams, etc.)
    eads = []
    dataset = json.load(open(dataset_path, 'r'))
    all_texts = [dataset[i]['text'] for i in range(len(dataset)) if dataset[i]['lang']=="en"]
    
    # 1. Determine Vocabulary size
    global_vocab = set() # determine V dynamically based on your specific dataset 
    
    for text in all_texts:
        tokens = word_tokenize(text.lower())
        global_vocab.update(tokens)
    
    V = len(global_vocab)

    for text in all_texts[0:subset]:   
        tokens = word_tokenize(text.lower())
        if len(tokens) < n:
            eads.append(0.0) 
            continue # Skip to next text
            
        # Get n-grams (or just tokens if n=1)
        if n == 1:
            items = tokens
        else:
            items = list(ngrams(tokens, n))
            
        C = len(items)          # Total number of tokens/n-grams
        N = len(set(items))     # Number of DISTINCT tokens/n-grams

        if C <= 0 or V <= 0:
            eads.append(0.0)
            continue
        
        # 2. Compute expectation E[N] = V * (1 - ((V-1)/V)^C)
        expected_N = V * (1 - math.pow((V - 1) / V, C))
        
        if expected_N == 0:
            eads.append(0.0)
        else:
            # 3. Compute EAD score for specific text
            ead_score = N / expected_N
            eads.append(ead_score)

    return eads

In [21]:
dataset_path = '/kaggle/input/dataset-full/dataset_full_200.json'
# 1. Run on the full English dataset (subset=200)
# This will return a list of 200 scores.
scores = compute_EAD(dataset_path, n=1, subset=200)

# 2. Split the results manually based on your dataset structure
# First 100 are Human, Last 100 are AI
human_scores = scores[:100]
ai_scores = scores[100:]

# 3. Print the final comparison
import numpy as np
print(f"Human English Mean: {np.mean(human_scores):.4f}")
print(f"AI English Mean:    {np.mean(ai_scores):.4f}")

Human English Mean: 0.4562
AI English Mean:    0.5111


## SBERT (Semantic Diversity)

### Steps & Formule
1. Split the text into sentences.
$S = \{s_1, ..., s_K\}$
3. Encode each sentence into a vector using a pre-trained SBERT model.
$u_i = \text{SBERT}(s_i)$
5. Calculate the cosine similarity between every possible pair of sentences.
$\text{sim}(u_i, u_j) = \frac{u_i \cdot u_j}{\|u_i\| \|u_j\|}$
7. Average these similarities and subtract from 1.
$\mu_{sim} = \frac{2}{K(K-1)} \sum_{1 \le i < j \le K} \text{sim}(u_i, u_j)$ & 
$\text{Diversity}_{\text{SBERT}} = 1 - \mu_{sim}$


Reference "[Text Similarity Implementation using BERT Embedding in Python](https://medium.com/@ahmedmellit/text-similarity-implementation-using-bert-embedding-in-python-1efdb5194e65)" by Medium: 


In [7]:
import numpy as np
import nltk
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure tokenizer is downloaded
nltk.download('punkt')

def compute_SBERTDiversity(dataset_path, language='en', model_name='all-MiniLM-L6-v2', subset=200):
    scores = []
    
    # 1. Load Dataset
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)
        
    # 2. Filter by Language ('en' or 'it')
    # This grabs both Human and AI texts for that language
    all_texts = [d['text'] for d in dataset if d['lang'] == language]

    print(f"Loading SBERT model: {model_name}...")
    model = SentenceTransformer(model_name)

    print(f"Processing {len(subset)} texts...")
    
    for text in all_texts[0:subset]:
        # 1. Split text into sentences
        # Use appropriate tokenizer for the language if possible, but default is usually fine
        sentences = nltk.sent_tokenize(text)
        
        # Edge case: Need at least 2 sentences to compare
        if len(sentences) < 2:
            scores.append(0.0)
            continue
            
        # 2. Encode sentences (Batch encoding is faster)
        embeddings = model.encode(sentences)
        
        # 3. Calculate Cosine Similarity Matrix
        sim_matrix = cosine_similarity(embeddings)
        
        # 4. Extract unique pairs (Upper Triangle)
        upper_tri_indices = np.triu_indices_from(sim_matrix, k=1)
        unique_similarities = sim_matrix[upper_tri_indices]
        
        # 5. Compute Diversity
        if len(unique_similarities) == 0:
            scores.append(0.0)
        else:
            mu_sim = np.mean(unique_similarities)
            diversity_score = 1 - mu_sim
            scores.append(diversity_score)
    
    return scores

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
dataset_path = '/kaggle/input/dataset-full/dataset_full_200.json'
# 1. Run on the full English dataset (subset=200)
# This will return a list of 200 scores.
scores = compute_SBERTDiversity(dataset_path, subset=10, model_name='all-MiniLM-L6-v2')
print(scores)

Loading SBERT model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing 200 texts...
[0.7913430631160736, 0.8134376257658005, 0.7941412627696991, 0.7179266214370728, 0.8515371680259705, 0.7241068780422211, 0.7656460106372833, 0.8376843482255936, 0.8483338803052902, 0.8339983969926834]
